# Hyperparameter Tests

## Datensatz importieren

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

df = pd.DataFrame(housing.data, columns=housing.feature_names)

df["Price"] = housing.target

df.head()

## Daten vorbereiten

In [ ]:
X = df.drop("Price", axis=1)
y = df["Price"]

print(X.head())
print(y.head())

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train_tf, X_test_tf, y_train_tf, y_test_tf = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train_tf.shape)
print(X_test_tf.shape)

## Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_tf = scaler.fit_transform(X_train_tf)
X_test_tf = scaler.transform(X_test_tf)

## Read Hyperparameter Settings from CSV

In [ ]:
param_df = pd.read_csv("Testfile.csv", sep=";", decimal=",")
print(param_df)

## Test Hyperparameters and save results to CSV

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

for index, row in param_df.iterrows():
    index_nr = row["Nr."]
    act_func = row["Activation Function"]
    layers = row["Anzahl Layer"]
    optimizer = row["Optimizer"]
    l_rate = row["Learning Rate"]
    loss_function = row["Loss Function"]
    epochs = row["Epochs"]
    batch_size = row["Batch Size"]
    

    if optimizer.lower() == "adam":
        ker_opt = keras.optimizers.Adam(learning_rate=l_rate)
    elif optimizer.lower() == "rmsprop":
        ker_opt = keras.optimizers.RMSprop(learning_rate=l_rate)
    elif optimizer.lower() == "sgd":
        ker_opt = keras.optimizers.SGD(learning_rate=l_rate)

    tf_model = tf.keras.Sequential()
    
    #Input Layer
    tf_model.add(keras.layers.Input(shape=(X_train_tf.shape[1],)))
    
    #Amount of layers
    for i in range(layers - 1):
        #Leaky ReLu muss anders angewendet werden als die übrigen Activation Functions
        if act_func == "Leaky ReLU":
            tf_model.add(keras.layers.Dense(64))
            tf_model.add(keras.layers.LeakyReLU())
        else:
            tf_model.add(keras.layers.Dense(64, activation=act_func))
    
    #Output Layer
    tf_model.add(keras.layers.Dense(1))
    
    tf_model.compile(
        optimizer=ker_opt,
        loss=loss_function,
        metrics=['mae']
    )
    
    history = tf_model.fit(
        X_train_tf,
        y_train_tf,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )
    
    # Vorhersagen mit dem neuen Modell
    y_pred = tf_model.predict(X_test_tf).reshape(-1)
    
    # Metriken berechnen
    mae_test = mean_absolute_error(y_test_tf, y_pred)
    mse_test = mean_squared_error(y_test_tf, y_pred)
    r2_test = r2_score(y_test_tf, y_pred)
    
    print("Test Model 1 - MAE:", mae_test)
    print("Test Model 1 - MSE:", mse_test)
    print("Test Model 1 - R2:", r2_test)

    results.append({
        "Nr.": index_nr,
        "MAE": mae_test,
        "MSE": mse_test,
        "R2": r2_test
    })

results_df = pd.DataFrame(results)

results_df.to_csv("test_results.csv", index=False)